## **Learning Objectives**

By completing these exercises, you will:

- Understand Retrieval-Augmented Generation (RAG) and its components.
- Load, preprocess, and handle PDF documents effectively.
- Convert textual data into embeddings for efficient retrieval.
- Implement and test document retrieval systems using LangChain and FAISS.
- Integrate retrieval systems with free Language Models (LLMs) from ChatGroq .
- Build an interactive chat-based Q&A system.

---

## **Exercise 1: Setup and Warm-up**

In this exercise, you'll set up your environment and select a suitable language model.

**Steps:**

1. **Load Environment Variables:** Ensure your environment variables (e.g., API keys, tokens) are securely stored and loaded.
2. **Choose LLM:** Select a free LLM model from from ChatGroq. 
3. **Instantiate the Model:** Create an instance of your chosen model.


In [ ]:
# Import necessary libraries
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint

# Load environment variables
load_dotenv()

/Users/asimeoa/aipm-1711/ds-rag-pipeline_sia/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

---

## **Exercise 2: Data Ingestion**

In this exercise, you'll learn to load PDF data into a Python environment.

**Steps:**

1. **Import PDF Loader:** Use LangChain’s `PyPDFLoader`.
2. **Load PDF File:** Create a function to read the PDF file.
3. **Display PDF Content:** Print the number of pages and first page content.

In [2]:
# Import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

# Example function to load PDF

def load_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

In [3]:
# Load your PDF and print out content here
file_path = "../documents/paracetamol.pdf"

In [4]:
# Load your PDF and print out content here
file_path = "../documents/paracetamol.pdf"

# Hier passiert die Magie: Wir rufen die Funktion auf und speichern das Ergebnis
documents = load_pdf(file_path)

In [5]:
# Kurzer Check, ob es geklappt hat
if documents:
    print(f"Erfolg! {len(documents)} Seiten geladen.")
    print(f"Vorschau der ersten Seite:\n{documents[0].page_content[:200]}...")

Erfolg! 3 Seiten geladen.
Vorschau der ersten Seite:
202211
178 mm
422 mm
178 mm
422 mm
Front Side Back Side
 Paracetamol 500mg Tablets
178 x 422mm
178 x 30mm
358
202211
NA
Printed Leaﬂet for  Paracetamol 500mg Tablets, Open size: 178 x 422mm, Folding S...


---

## **Exercise 3: Document Chunking**

This exercise introduces splitting large documents into manageable text chunks.

**Steps:**

1. **Import Text Splitter:** Use `RecursiveCharacterTextSplitter`.
2. **Chunk Document:** Write a function that splits loaded documents into chunks.
3. **Test Function:** Verify by displaying the resulting chunks.


In [6]:
# Import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Example chunking function
def chunk_documents(documents, chunk_size=500, chunk_overlap=50):
    # 1. Den "Zerstückler" einstellen
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True
    )
    
    # 2. Den Splitter auf die Dokumente ANWENDEN
    chunks = text_splitter.split_documents(documents)
    
    # 3. Das Ergebnis ZURÜCKGEBEN
    return chunks

In [7]:
# Execute your chunking function here
chunks = chunk_documents(documents)

# Check, ob es geklappt hat
print(f"Anzahl der erzeugten Chunks: {len(chunks)}")

Anzahl der erzeugten Chunks: 45


In [8]:
# --- NEUE ZELLE NACH EXERCISE 3 ---

import os
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

# 1. Pfad festlegen (einen Ordner höher als dein aktuelles Verzeichnis)
persist_directory = '../chroma_db' 

# 2. Embeddings initialisieren (das "Übersetzungs-Tool" für Ollama)
embeddings = OllamaEmbeddings(model="llama3.2:3b")

print("Erstelle Vektor-Datenbank... Bitte warten (Ollama arbeitet)...")

# 3. Die Datenbank aus den 'chunks' von oben erstellen
vector_db = Chroma.from_documents(
    documents=chunks, # Wichtig: Hier nutzen wir deine Variable 'chunks' aus Exercise 3
    embedding=embeddings,
    persist_directory=persist_directory
)

# 4. Dauerhaft auf der Festplatte speichern
vector_db.persist()

print(f"✅ ERFOLG: Die Datenbank wurde gespeichert unter: {os.path.abspath(persist_directory)}")
print(f"Vektor-Datenbank ist bereit! Anzahl der Dokumente: {vector_db._collection.count()}")

Erstelle Vektor-Datenbank... Bitte warten (Ollama arbeitet)...


/var/folders/_v/rjrhrzpx5l1bcyfw7cznsxsw0000gn/T/ipykernel_26573/447879557.py:11: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="llama3.2:3b")


✅ ERFOLG: Die Datenbank wurde gespeichert unter: /Users/asimeoa/aipm-1711/ds-rag-pipeline_sia/chroma_db
Vektor-Datenbank ist bereit! Anzahl der Dokumente: 135


/var/folders/_v/rjrhrzpx5l1bcyfw7cznsxsw0000gn/T/ipykernel_26573/447879557.py:23: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()



---

## **Exercise 4: Embedding and Storage**

In this exercise, you will create embeddings from text chunks and store them efficiently.

**Steps:**

1. **Choose Embedding Model:** Use `sentence-transformers/all-mpnet-base-v2` from Hugging Face.
2. **Generate Embeddings:** Transform document chunks into embeddings.
3. **Store Embeddings:** Save these embeddings using FAISS locally.


In [9]:
# Import libraries
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Example function for embeddings and storage
def embed_and_store(chunks):
    # 1. Embeddings erstellen
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
    
    # 2. Vektorstore erstellen und mit Chunks füllen
    vectorstore = FAISS.from_documents(chunks, embeddings)
    
    # 3. Vektorstore speichern (optional)
    vectorstore.save_local("faiss_index")
    
    return vectorstore

In [10]:
# Generate embeddings and save them locally
vector_db = embed_and_store(chunks)

In [ ]:
# Prüfen, ob vector_db erstellt wurde
if vector_db:
    print("Check pass: Vektor data saved!")
    
# Prüfen, ob der Ordner auf der Festplatte erstellt wurde
import os
if os.path.exists("faiss_index"):
    print("Check pass: Folder 'faiss_index' was created locally!")

Check pass: Vektor data saved!
Check pass: Folder 'faiss_index' was created locally!


---

## **Exercise 5: Retrieval from FAISS**

Here, you will learn how to retrieve documents from a vector database using embeddings.

**Steps:**

1. **Load Embeddings:** Load stored embeddings from the FAISS database.
2. **Implement Retrieval:** Create logic to retrieve relevant chunks based on queries.
3. **Test Retriever:** Execute retrieval using sample queries.

In [12]:
# Implement retrieval logic from your FAISS database
# Load the FAISS database
vector_db = FAISS.load_local(
    "faiss_index", 
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2"),
    allow_dangerous_deserialization=True)

# Example retrieval function
def retrieve_documents(query, k=5):
    return vector_db.similarity_search(query, k=k)

In [13]:
# Test your retrieval system with queries
test_query = "What is paracetamol used for?"
retrieved_docs = retrieve_documents(test_query) 

print(f"Suche nach: {test_query}\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Treffer {i+1} ---")
    print(doc.page_content)
    print("\n")

Suche nach: What is paracetamol used for?

--- Treffer 1 ---
202211
178 mm
422 mm
178 mm
422 mm
Front Side Back Side
 Paracetamol 500mg Tablets
178 x 422mm
178 x 30mm
358
202211
NA
Printed Leaﬂet for  Paracetamol 500mg Tablets, Open size: 178 x 422mm, Folding Size : 178x30mm 
Speciﬁcation: 40GSM Bible Paper - Fairmed/Apohilft-Germany 
P4S Complete Solutions
01
Black
Fairmed/Apohilft-Germany 
30mm
PLPARA500DESKRNOW1.1
Stillzeit
Paracetamol geht in die Muttermilch über. Da nachteilige Folgen für


--- Treffer 2 ---
202211
178 mm
422 mm
178 mm
422 mm
Front Side Back Side
 Paracetamol 500mg Tablets
178 x 422mm
178 x 30mm
358
202211
NA
Printed Leaﬂet for  Paracetamol 500mg Tablets, Open size: 178 x 422mm, Folding Size : 178x30mm 
Speciﬁcation: 40GSM Bible Paper - Fairmed/Apohilft-Germany 
P4S Complete Solutions
01
Black
Fairmed/Apohilft-Germany 
30mm
Gebrauchsinformation: Information für den Anwender
Paracetamol 500 mg Die Apotheke hilft 
Schmerztabletten


--- Treffer 3 ---
von Paracetamol

---

## **Exercise 6: Connecting Retrieval with LLM**

You'll now connect document retrieval with the Language Model.

**Steps:**

1. **Create Retrieval Chain:** Link your retrieval system to your instantiated LLM.
2. **Test the Chain:** Confirm it works by generating answers from retrieved documents.

In [14]:
from langchain.chains import RetrievalQA

def create_retrieval_chain(vector_db, prompt_template, model):
    # This searches for the 5 best text parts in your PDF
    retriever = vector_db.as_retriever(search_kwargs={"k": 5})
    
    # This builds the final chain
    chain = RetrievalQA.from_chain_type(
        llm=model,
        chain_type="stuff",
        retriever=retriever,
        input_key="query",  # This must be "query"
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt_template}
    )
    return chain

In [15]:
from langchain.prompts import PromptTemplate

# Simple English instructions for the AI
template = """You are a helpful medical assistant. 
Use only the context below to answer the question. 
If you don't know the answer, just say you don't know.

Context: {context}
Question: {query}

Answer:"""

rag_prompt = PromptTemplate(
    template=template, 
    input_variables=["context", "query"]
)
print("✅ Step 2 complete: Prompt is ready!")

✅ Step 2 complete: Prompt is ready!


In [16]:
from langchain_community.chat_models import ChatOllama

# 1. Load your local Ollama model
model = ChatOllama(model="llama3.2:3b") 

# 2. Build the actual chain using the function from Step 1
# 'vector_db' must already exist from your previous exercise
rag_chain = create_retrieval_chain(vector_db, rag_prompt, model)

print("✅ Step 3 complete: AI Chain is built!")

✅ Step 3 complete: AI Chain is built!


/var/folders/_v/rjrhrzpx5l1bcyfw7cznsxsw0000gn/T/ipykernel_26573/2255243219.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  model = ChatOllama(model="llama3.2:3b")


In [17]:
# The key here MUST be "query" to match Step 1 and Step 2
sample_query = "What is the recommended dosage for children?"

print("Sending question to Ollama... please wait...")

try:
    response = rag_chain.invoke({"query": sample_query})
    print("\n--- AI Response ---")
    print(response["result"])
except Exception as e:
    print(f"\n❌ Error: {e}")

Sending question to Ollama... please wait...

❌ Error: Missing some input keys: {'query'}


---

## **Exercise 7: Interactive Chat System**

In the final exercise, build an interactive chat-based query system.

**Steps:**

1. **Create Chat Interface:** Develop a simple function for interactive querying.
2. **Run the Chat:** Allow users to ask questions and receive immediate responses.


In [ ]:
def interactive_chat(chain):
    print("--- Medical AI Chat (Type 'exit' to stop) ---")
    
    while True:
        # User input
        user_input = input("Your Question: ")
        
        # Stop condition
        if user_input.lower() in ["exit", "quit", "stop"]:
            print("Chat ended. Goodbye!")
            break
            
        # Process the question
        try:
            print("AI is thinking...")
            response = chain.invoke({"query": user_input})
            
            print("\n" + "="*30)
            print("AI RESPONSE:")
            print(response["result"])
            print("="*30 + "\n")
            
        except Exception as e:
            print(f"An error occurred: {e}")

# Start the chat
interactive_chat(rag_chain)

--- Medical AI Chat (Type 'exit' to stop) ---


---

## **Conclusion & Reflection**

After completing these exercises:

- Summarize key concepts learned.
- Reflect on the effectiveness and limitations of the free LLM and RAG system you've built.
- Consider how you might improve or extend your system in practical applications.

---